## Ejercicio 2: Escalamiento de tickets de soporte técnico

**Curso:** Inteligencia Artificial I (IS-484) · **Alumno:** Henry Josue Flores Saras

### Ficha PEAS

| Componente | Descripción |
|---|---|
| **Percepción (S)** | `tiempo_espera_minutos` (entero), `nivel_urgencia` ("baja", "media", "alta"), `cliente_premium` (True/False). |
| **Acciones (A)** | `mantener en nivel 1` (soporte básico), `escalar a nivel 2` (técnico especializado), `escalar a nivel 3` (ingeniería/expertos), `revisar clasificacion` (dato inválido). |
| **Entorno (E)** | Mesa de ayuda con una cola de tickets, clientes estándar y premium, niveles de soporte con capacidad limitada y acuerdos de nivel de servicio (SLA). |
| **Objetivo** | Atender cada ticket en el nivel adecuado: resolver rápido lo crítico sin saturar a los niveles superiores con casos simples. |
| **Medida de desempeño (P)** | Tiempo promedio de resolución, % de tickets que cumplen el SLA (en especial premium), satisfacción del cliente y carga de los niveles 2 y 3 (pocos escalamientos innecesarios). |

### Justificación de las reglas

Una urgencia alta implica un servicio caído o una pérdida para el cliente, por eso se escala de inmediato al nivel 3 sin importar la espera. En urgencia media y baja el tiempo de espera actúa como un "reloj de SLA": cuanto más espera el cliente, más probable es que el nivel 1 no pueda resolverlo y crezca su insatisfacción, así que se escala al nivel 2 al superar un umbral. Los clientes premium tienen un SLA más exigente, por lo que su umbral es la mitad (30 vs. 60 min en media; 120 vs. 240 min en baja). Así el cliente no queda olvidado en la cola y el equipo reserva los niveles 2 y 3 para lo que de verdad lo necesita.

### Código

In [1]:
# Umbrales de espera (minutos) para escalar a nivel 2: (premium, estandar)
UMBRAL_MEDIA = {True: 30, False: 60}
UMBRAL_BAJA = {True: 120, False: 240}


def agente_soporte(tiempo_espera_minutos, nivel_urgencia, cliente_premium):
    """Agente reactivo simple: decide el nivel de soporte y retorna (accion, motivo)."""
    tipo = "premium" if cliente_premium else "estandar"

    # Validacion de percepciones
    if not isinstance(tiempo_espera_minutos, int) or tiempo_espera_minutos < 0:
        return "revisar clasificacion", f"tiempo de espera no valido ({tiempo_espera_minutos})"
    urgencia = str(nivel_urgencia).strip().lower()
    if urgencia not in ("baja", "media", "alta"):
        return "revisar clasificacion", f"nivel de urgencia desconocido ('{nivel_urgencia}')"

    # Regla 1: urgencia alta -> nivel 3 inmediato
    if urgencia == "alta":
        return "escalar a nivel 3", f"urgencia alta reportada por cliente {tipo}, se atiende de inmediato"

    # Regla 2: urgencia media -> depende de espera y tipo de cliente
    if urgencia == "media":
        umbral = UMBRAL_MEDIA[cliente_premium]
        if tiempo_espera_minutos >= umbral:
            return ("escalar a nivel 2",
                    f"urgencia media y cliente {tipo} con {tiempo_espera_minutos} min de espera (umbral {umbral} min)")
        return ("mantener en nivel 1",
                f"urgencia media, cliente {tipo} con {tiempo_espera_minutos} min, aun dentro del umbral de {umbral} min")

    # Regla 3: urgencia baja -> solo escala si la espera es muy larga
    umbral = UMBRAL_BAJA[cliente_premium]
    if tiempo_espera_minutos >= umbral:
        return ("escalar a nivel 2",
                f"urgencia baja pero cliente {tipo} lleva {tiempo_espera_minutos} min esperando (umbral {umbral} min)")
    return ("mantener en nivel 1",
            f"urgencia baja, cliente {tipo} con {tiempo_espera_minutos} min, dentro del umbral de {umbral} min")

### Simulación y pruebas